In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/psrc0514/wifi-identification/wifi_measurements.csv


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

df = pd.read_csv("/kaggle/input/datasets/psrc0514/wifi-identification/wifi_measurements.csv")

print(df.shape)
display(df.head())

(429, 12)


,Session,Location,Crowd Density,Time,BSSID,Signal (%),Signal (dBm),Ping (ms),Link (Mbps),DL (Mbps),UL (Mbps),Status
0,NIGHT,DISANG ROAD,1,0:25:24,Unknown,92,-54,525,72.2,0.0,0.0,DEGRADED
1,NIGHT,DISANG ROAD,1,0:25:30,Unknown,92,-54,152,72.2,0.0,0.0,DEGRADED
2,NIGHT,DISANG ROAD,1,0:25:36,Unknown,92,-54,553,72.2,0.0,0.0,DEGRADED
3,NIGHT,DISANG ROAD,1,0:25:42,Unknown,92,-54,356,72.2,0.0,0.0,DEGRADED
4,NIGHT,DISANG ROAD,1,0:25:47,Unknown,92,-54,399,72.2,0.0,0.0,DEGRADED


In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

df.rename(
    columns={"signal_%": "signal_percent"},
    inplace=True
)

numeric_columns = [
    "crowd_density",
    "signal_percent",
    "signal_dbm",
    "ping_ms",
    "link_mbps",
    "dl_mbps",
    "ul_mbps"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [5]:
print(df["dl_mbps"].describe())

count    429.000000
mean      19.911678
std       33.777037
min        0.000000
25%        0.000000
50%        5.000000
75%       19.200000
max      189.590000
Name: dl_mbps, dtype: float64


In [6]:
print(
    df["dl_mbps"].quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

0.10     0.00
0.25     0.00
0.50     5.00
0.75    19.20
0.90    79.95
Name: dl_mbps, dtype: float64


In [7]:
print(df["ping_ms"].describe())

count     429.000000
mean      749.067599
std       399.127468
min        54.000000
25%       238.000000
50%       999.000000
75%       999.000000
max      1638.000000
Name: ping_ms, dtype: float64


In [8]:
df["ping_failed"] = (
    df["ping_ms"] >= 999
).astype(int)

df["ping_clean"] = df["ping_ms"].replace(
    999,
    np.nan
)

In [9]:
print(df["dl_mbps"].describe())

print("\nPercentiles:")
print(
    df["dl_mbps"].quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\nPing:")
print(df["ping_ms"].describe())

count    429.000000
mean      19.911678
std       33.777037
min        0.000000
25%        0.000000
50%        5.000000
75%       19.200000
max      189.590000
Name: dl_mbps, dtype: float64

Percentiles:
0.10     0.00
0.25     0.00
0.50     5.00
0.75    19.20
0.90    79.95
Name: dl_mbps, dtype: float64

Ping:
count     429.000000
mean      749.067599
std       399.127468
min        54.000000
25%       238.000000
50%       999.000000
75%       999.000000
max      1638.000000
Name: ping_ms, dtype: float64


In [10]:
print("Ping failures:")
print(df["ping_failed"].value_counts())

print("\nPing failure percentage:")
print(df["ping_failed"].mean() * 100)

Ping failures:
ping_failed
1    300
0    129
Name: count, dtype: int64

Ping failure percentage:
69.93006993006993


In [11]:
# Temporary threshold - we will finalize this after checking the data
DOWNLOAD_THRESHOLD = 5

In [12]:
df["poor_connectivity"] = (
    (df["dl_mbps"] <= DOWNLOAD_THRESHOLD) |
    (df["ping_failed"] == 1)
).astype(int)

In [13]:
print("Poor connectivity distribution:")
print(df["poor_connectivity"].value_counts())

print("\nPercentage:")
print(
    df["poor_connectivity"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Poor connectivity distribution:
poor_connectivity
1    363
0     66
Name: count, dtype: int64

Percentage:
poor_connectivity
1    84.62
0    15.38
Name: proportion, dtype: float64


In [14]:
print(
    df["poor_connectivity"].value_counts()
)

print(
    df["poor_connectivity"].value_counts(
        normalize=True
    ) * 100
)

poor_connectivity
1    363
0     66
Name: count, dtype: int64
poor_connectivity
1    84.615385
0    15.384615
Name: proportion, dtype: float64


In [15]:
df["hour"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour

In [16]:
df["minute"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.minute

In [17]:
df["minute"] = pd.to_datetime(
    df["time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.minute

In [18]:
features = [
    "location",
    "session",
    "hour",
    "crowd_density",
    "signal_dbm"
]

In [19]:
X = df[features]
y = df["poor_connectivity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [20]:
categorical_features = [
    "location",
    "session"
]

numerical_features = [
    "hour",
    "crowd_density",
    "signal_dbm"
]

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numerical_features
        )
    ]
)

In [22]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [23]:
logistic_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['location', 'session']),
                                                 ('num', StandardScaler(),
                                                  ['hour', 'crowd_density',
                                                   'signal_dbm'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [24]:
y_pred = logistic_model.predict(X_test)

y_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

In [25]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

              precision    recall  f1-score   support

           0       0.65      1.00      0.79        13
           1       1.00      0.90      0.95        73

    accuracy                           0.92        86
   macro avg       0.82      0.95      0.87        86
weighted avg       0.95      0.92      0.93        86

ROC-AUC: 0.9452054794520548


In [26]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [27]:
rf_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['location', 'session']),
                                                 ('num', StandardScaler(),
                                                  ['hour', 'crowd_density',
                                                   'signal_dbm'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=8,
                                        min_samples_leaf=3, n_estimators=300,
                                        random_state=42))])

In [28]:
rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

In [29]:
print(
    classification_report(
        y_test,
        rf_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        rf_prob
    )
)

              precision    recall  f1-score   support

           0       0.68      1.00      0.81        13
           1       1.00      0.92      0.96        73

    accuracy                           0.93        86
   macro avg       0.84      0.96      0.88        86
weighted avg       0.95      0.93      0.94        86

ROC-AUC: 0.9831401475237092


In [30]:
def predict_wifi_risk(
    model,
    location,
    session,
    hour,
    crowd_density,
    signal_dbm
):
    input_data = pd.DataFrame({
        "location": [location],
        "session": [session],
        "hour": [hour],
        "crowd_density": [crowd_density],
        "signal_dbm": [signal_dbm]
    })

    probability = model.predict_proba(
        input_data
    )[0, 1]

    prediction = (
        "HIGH RISK"
        if probability >= 0.5
        else "LOW RISK"
    )

    return prediction, probability

In [31]:
prediction, probability = predict_wifi_risk(
    rf_model,
    location="New SAC",
    session="MORNING",
    hour=10,
    crowd_density=4,
    signal_dbm=-58
)

print("Prediction:", prediction)
print(
    f"Risk probability: {probability*100:.2f}%"
)

Prediction: HIGH RISK
Risk probability: 100.00%


In [32]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, rf_prob)
    ]
})

display(results.round(3))

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.919,1.0,0.904,0.950,0.945
1,Random Forest,0.930,1.0,0.918,0.957,0.983


In [33]:
print(df.shape)
print(df.columns.tolist())

(429, 17)
['session', 'location', 'crowd_density', 'time', 'bssid', 'signal_percent', 'signal_dbm', 'ping_ms', 'link_mbps', 'dl_mbps', 'ul_mbps', 'status', 'ping_failed', 'ping_clean', 'poor_connectivity', 'hour', 'minute']


In [34]:
print(sorted(df["location"].dropna().unique()))

['ADMIN BUILDING', 'CANTEEN', 'CLASSROOM', 'CORE5 GF', 'DISANG ROAD', 'HOSTEL ROOM', 'LIBRARY', 'NEW SAC', 'OPEN ROAD', 'ROADS', 'VISITOR ROOM', 'new sac']


In [35]:
df["location"] = (
    df["location"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(sorted(df["location"].unique()))

['ADMIN BUILDING', 'CANTEEN', 'CLASSROOM', 'CORE5 GF', 'DISANG ROAD', 'HOSTEL ROOM', 'LIBRARY', 'NEW SAC', 'OPEN ROAD', 'ROADS', 'VISITOR ROOM']


In [36]:
network_profile = (
    df.groupby("location")
      .agg(
          measurements=("location", "size"),
          median_signal=("signal_dbm", "median"),
          median_download=("dl_mbps", "median"),
          ping_failure_rate=("ping_failed", "mean"),
          median_crowd=("crowd_density", "median")
      )
      .reset_index()
)

network_profile["ping_failure_rate"] *= 100

display(network_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd
0,ADMIN BUILDING,39,-60.0,40.46,38.461538,1.0
1,CANTEEN,26,-81.0,0.34,84.615385,2.0
2,CLASSROOM,65,-58.0,8.67,100.000000,2.0
3,CORE5 GF,64,-59.0,9.07,100.000000,5.0
4,DISANG ROAD,12,-54.0,4.78,0.000000,1.0
5,HOSTEL ROOM,10,-55.0,5.19,100.000000,3.0
6,LIBRARY,20,-63.5,0.34,100.000000,3.0
7,NEW SAC,101,-57.0,0.00,63.366337,4.0
8,OPEN ROAD,23,-83.0,0.00,100.000000,1.0
9,ROADS,55,-53.0,62.03,5.454545,1.0


In [37]:
network_profile.to_csv(
    "network_profile.csv",
    index=False
)

print("Saved network_profile.csv")

Saved network_profile.csv


In [39]:
import pandas as pd

survey_location_profile = pd.read_csv(
    "/kaggle/input/datasets/psrc0514/survey-location/survey_location_profile.csv"
)

display(survey_location_profile)

,location_group,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [40]:
location_mapping = {
    "HOSTEL ROOM": "HOSTEL ROOM",
    "CLASSROOM": "ACADEMIC BUILDINGS",
    "LIBRARY": "LIBRARY",
    "CANTEEN": "MESS / CANTEEN"
}

network_profile["survey_category"] = (
    network_profile["location"].map(location_mapping)
)

display(network_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category
0,ADMIN BUILDING,39,-60.0,40.46,38.461538,1.0,NaN
1,CANTEEN,26,-81.0,0.34,84.615385,2.0,MESS / CANTEEN
2,CLASSROOM,65,-58.0,8.67,100.000000,2.0,ACADEMIC BUILDINGS
3,CORE5 GF,64,-59.0,9.07,100.000000,5.0,NaN
4,DISANG ROAD,12,-54.0,4.78,0.000000,1.0,NaN
5,HOSTEL ROOM,10,-55.0,5.19,100.000000,3.0,HOSTEL ROOM
6,LIBRARY,20,-63.5,0.34,100.000000,3.0,LIBRARY
7,NEW SAC,101,-57.0,0.00,63.366337,4.0,NaN
8,OPEN ROAD,23,-83.0,0.00,100.000000,1.0,NaN
9,ROADS,55,-53.0,62.03,5.454545,1.0,NaN


In [41]:
combined_profile = network_profile.merge(
    survey_location_profile,
    left_on="survey_category",
    right_on="location_group",
    how="left"
)

display(combined_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category,location_group,survey_mentions
0,ADMIN BUILDING,39,-60.0,40.46,38.461538,1.0,NaN,NaN,NaN
1,CANTEEN,26,-81.0,0.34,84.615385,2.0,MESS / CANTEEN,MESS / CANTEEN,6.0
2,CLASSROOM,65,-58.0,8.67,100.000000,2.0,ACADEMIC BUILDINGS,ACADEMIC BUILDINGS,13.0
3,CORE5 GF,64,-59.0,9.07,100.000000,5.0,NaN,NaN,NaN
4,DISANG ROAD,12,-54.0,4.78,0.000000,1.0,NaN,NaN,NaN
5,HOSTEL ROOM,10,-55.0,5.19,100.000000,3.0,HOSTEL ROOM,HOSTEL ROOM,49.0
6,LIBRARY,20,-63.5,0.34,100.000000,3.0,LIBRARY,LIBRARY,7.0
7,NEW SAC,101,-57.0,0.00,63.366337,4.0,NaN,NaN,NaN
8,OPEN ROAD,23,-83.0,0.00,100.000000,1.0,NaN,NaN,NaN
9,ROADS,55,-53.0,62.03,5.454545,1.0,NaN,NaN,NaN


In [42]:
print(df.columns.tolist())

['session', 'location', 'crowd_density', 'time', 'bssid', 'signal_percent', 'signal_dbm', 'ping_ms', 'link_mbps', 'dl_mbps', 'ul_mbps', 'status', 'ping_failed', 'ping_clean', 'poor_connectivity', 'hour', 'minute']


In [43]:
print(X_train.columns.tolist())

['location', 'session', 'hour', 'crowd_density', 'signal_dbm']


In [44]:
print(df[["location", "crowd_density", "dl_mbps", "ping_failed"]].head())

      location  crowd_density  dl_mbps  ping_failed
0  DISANG ROAD              1      0.0            0
1  DISANG ROAD              1      0.0            0
2  DISANG ROAD              1      0.0            0
3  DISANG ROAD              1      0.0            0
4  DISANG ROAD              1      0.0            0


In [45]:
print("Models / important variables currently available:")

for name in [
    "model",
    "rf_model",
    "random_forest",
    "clf",
    "pipeline",
    "rf",
    "X_train",
    "y_train"
]:
    if name in globals():
        print(name, "->", type(globals()[name]))

Models / important variables currently available:
rf_model -> <class 'sklearn.pipeline.Pipeline'>
X_train -> <class 'pandas.core.frame.DataFrame'>
y_train -> <class 'pandas.core.series.Series'>


In [49]:
print(df["session"].unique())

['NIGHT' 'MORNING']


In [50]:
print(df["hour"].min(), df["hour"].max())

0 14


In [51]:
signal_lookup = (
    df.groupby("location")["signal_dbm"]
      .median()
      .to_dict()
)

print(signal_lookup)

{'ADMIN BUILDING': -60.0, 'CANTEEN': -81.0, 'CLASSROOM': -58.0, 'CORE5 GF': -59.0, 'DISANG ROAD': -54.0, 'HOSTEL ROOM': -55.0, 'LIBRARY': -63.5, 'NEW SAC': -57.0, 'OPEN ROAD': -83.0, 'ROADS': -53.0, 'VISITOR ROOM': -58.0}


In [53]:
import pandas as pd
from datetime import datetime

def predict_wifi(location, time, crowd):
    
    # -------------------------
    # 1. Clean location
    # -------------------------
    location = str(location).strip().upper()
    
    # Check whether location exists
    if location not in signal_lookup:
        return {
            "error": f"No historical data available for location: {location}",
            "available_locations": sorted(signal_lookup.keys())
        }
    
    # -------------------------
    # 2. Convert time to hour
    # -------------------------
    try:
        parsed_time = pd.to_datetime(time)
        hour = parsed_time.hour
    except:
        return {
            "error": "Invalid time format. Try something like '9:00 PM' or '21:00'."
        }
    
    # -------------------------
    # 3. Determine session
    # -------------------------
    if hour >= 18 or hour < 6:
        session = "NIGHT"
    else:
        session = "MORNING"
    
    # -------------------------
    # 4. Get historical signal
    # -------------------------
    signal_dbm = signal_lookup[location]
    
    # -------------------------
    # 5. Create model input
    # -------------------------
    input_data = pd.DataFrame([{
        "location": location,
        "session": session,
        "hour": hour,
        "crowd_density": crowd,
        "signal_dbm": signal_dbm
    }])
    
    # -------------------------
    # 6. Predict
    # -------------------------
    prediction = rf_model.predict(input_data)[0]
    
    probability = None
    
    if hasattr(rf_model, "predict_proba"):
        probabilities = rf_model.predict_proba(input_data)[0]
        
        # Find probability of poor connectivity = 1
        classes = rf_model.classes_
        
        if 1 in classes:
            poor_index = list(classes).index(1)
            probability = probabilities[poor_index]
    
    # -------------------------
    # 7. Risk label
    # -------------------------
    if probability is not None:
        if probability >= 0.70:
            risk = "HIGH"
        elif probability >= 0.40:
            risk = "MEDIUM"
        else:
            risk = "LOW"
    else:
        risk = "POOR CONNECTIVITY" if prediction == 1 else "GOOD CONNECTIVITY"
    
    return {
        "location": location,
        "time": time,
        "hour": hour,
        "session": session,
        "crowd": crowd,
        "historical_signal_dbm": signal_dbm,
        "prediction": int(prediction),
        "poor_connectivity_probability": probability,
        "risk": risk
    }

In [54]:
result = predict_wifi(
    location="NEW SAC",
    time="9:00 PM",
    crowd=4
)

print(result)

{'location': 'NEW SAC', 'time': '9:00 PM', 'hour': 21, 'session': 'NIGHT', 'crowd': 4, 'historical_signal_dbm': -57.0, 'prediction': 1, 'poor_connectivity_probability': np.float64(0.8668310805021265), 'risk': 'HIGH'}


In [55]:
tests = [
    ("NEW SAC", "9:00 PM", 4),
    ("LIBRARY", "9:00 PM", 3),
    ("CANTEEN", "1:00 PM", 2),
    ("CLASSROOM", "10:00 AM", 5),
    ("HOSTEL ROOM", "11:00 PM", 3)
]

for location, time, crowd in tests:
    
    result = predict_wifi(
        location=location,
        time=time,
        crowd=crowd
    )
    
    print("\n----------------------------")
    print(f"Location : {location}")
    print(f"Time     : {time}")
    print(f"Crowd    : {crowd}")
    print(f"Risk     : {result.get('risk')}")
    print(f"Probability : {result.get('poor_connectivity_probability')}")


----------------------------
Location : NEW SAC
Time     : 9:00 PM
Crowd    : 4
Risk     : HIGH
Probability : 0.8668310805021265

----------------------------
Location : LIBRARY
Time     : 9:00 PM
Crowd    : 3
Risk     : HIGH
Probability : 0.8247653813775464

----------------------------
Location : CANTEEN
Time     : 1:00 PM
Crowd    : 2
Risk     : HIGH
Probability : 1.0

----------------------------
Location : CLASSROOM
Time     : 10:00 AM
Crowd    : 5
Risk     : HIGH
Probability : 1.0

----------------------------
Location : HOSTEL ROOM
Time     : 11:00 PM
Crowd    : 3
Risk     : HIGH
Probability : 0.7367931696880449


In [61]:
print(df["poor_connectivity"].value_counts())
print()
print(df["poor_connectivity"].value_counts(normalize=True) * 100)

poor_connectivity
1    363
0     66
Name: count, dtype: int64

poor_connectivity
1    84.615385
0    15.384615
Name: proportion, dtype: float64


In [62]:
actual_predictions = rf_model.predict(df[
    ["location", "session", "hour", "crowd_density", "signal_dbm"]
])

print("Actual poor connectivity:")
print(df["poor_connectivity"].mean())

print("\nModel predicted poor connectivity:")
print(actual_predictions.mean())

print("\nPrediction counts:")
print(pd.Series(actual_predictions).value_counts())

Actual poor connectivity:
0.8461538461538461

Model predicted poor connectivity:
0.7575757575757576

Prediction counts:
1    325
0    104
Name: count, dtype: int64


In [63]:
for crowd in range(1, 6):

    result = predict_wifi(
        location="NEW SAC",
        time="9:00 PM",
        crowd=crowd
    )

    print(
        f"Crowd {crowd}: "
        f"{result['poor_connectivity_probability']:.2%} "
        f"({result['risk']})"
    )

Crowd 1: 54.57% (MEDIUM)
Crowd 2: 78.44% (HIGH)
Crowd 3: 81.32% (HIGH)
Crowd 4: 86.68% (HIGH)
Crowd 5: 86.68% (HIGH)


In [64]:
print(network_profile.columns.tolist())

['location', 'measurements', 'median_signal', 'median_download', 'ping_failure_rate', 'median_crowd', 'survey_category']


In [65]:
display(network_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category
0,ADMIN BUILDING,39,-60.0,40.46,38.461538,1.0,NaN
1,CANTEEN,26,-81.0,0.34,84.615385,2.0,MESS / CANTEEN
2,CLASSROOM,65,-58.0,8.67,100.000000,2.0,ACADEMIC BUILDINGS
3,CORE5 GF,64,-59.0,9.07,100.000000,5.0,NaN
4,DISANG ROAD,12,-54.0,4.78,0.000000,1.0,NaN
5,HOSTEL ROOM,10,-55.0,5.19,100.000000,3.0,HOSTEL ROOM
6,LIBRARY,20,-63.5,0.34,100.000000,3.0,LIBRARY
7,NEW SAC,101,-57.0,0.00,63.366337,4.0,NaN
8,OPEN ROAD,23,-83.0,0.00,100.000000,1.0,NaN
9,ROADS,55,-53.0,62.03,5.454545,1.0,NaN


In [68]:
# Create a clean survey evidence table
survey_evidence = survey_location_profile.copy()

# Rename columns to make them easier to work with
survey_evidence.columns = [
    "survey_category",
    "survey_mentions"
]

display(survey_evidence)

,survey_category,survey_mentions
0,HOSTEL ROOM,49
1,ACADEMIC BUILDINGS,13
2,HOSTEL COMMON AREA,12
3,LIBRARY,7
4,MESS / CANTEEN,6
5,OTHER,3


In [69]:
combined_profile = network_profile.merge(
    survey_evidence,
    on="survey_category",
    how="left"
)

display(combined_profile)

,location,measurements,median_signal,median_download,ping_failure_rate,median_crowd,survey_category,survey_mentions
0,ADMIN BUILDING,39,-60.0,40.46,38.461538,1.0,NaN,NaN
1,CANTEEN,26,-81.0,0.34,84.615385,2.0,MESS / CANTEEN,6.0
2,CLASSROOM,65,-58.0,8.67,100.000000,2.0,ACADEMIC BUILDINGS,13.0
3,CORE5 GF,64,-59.0,9.07,100.000000,5.0,NaN,NaN
4,DISANG ROAD,12,-54.0,4.78,0.000000,1.0,NaN,NaN
5,HOSTEL ROOM,10,-55.0,5.19,100.000000,3.0,HOSTEL ROOM,49.0
6,LIBRARY,20,-63.5,0.34,100.000000,3.0,LIBRARY,7.0
7,NEW SAC,101,-57.0,0.00,63.366337,4.0,NaN,NaN
8,OPEN ROAD,23,-83.0,0.00,100.000000,1.0,NaN,NaN
9,ROADS,55,-53.0,62.03,5.454545,1.0,NaN,NaN


In [71]:
def get_wifi_prediction(location, time, crowd):
    
    # Get the existing ML prediction
    prediction = predict_wifi(
        location=location,
        time=time,
        crowd=crowd
    )
    
    # Stop if prediction failed
    if "error" in prediction:
        return prediction
    
    # Find physical network profile
    network_info = network_profile[
        network_profile["location"] == prediction["location"]
    ]
    
    if len(network_info) > 0:
        network_info = network_info.iloc[0]
        
        prediction["median_download_mbps"] = network_info["median_download"]
        prediction["ping_failure_rate"] = network_info["ping_failure_rate"]
        prediction["median_crowd"] = network_info["median_crowd"]
        prediction["survey_category"] = network_info["survey_category"]
        
    else:
        prediction["median_download_mbps"] = None
        prediction["ping_failure_rate"] = None
        prediction["median_crowd"] = None
        prediction["survey_category"] = None
    
    # Find survey evidence
    survey_category = prediction.get("survey_category")
    
    if pd.notna(survey_category):
        
        survey_match = survey_evidence[
            survey_evidence["survey_category"] == survey_category
        ]
        
        if len(survey_match) > 0:
            prediction["survey_mentions"] = int(
                survey_match.iloc[0]["survey_mentions"]
            )
        else:
            prediction["survey_mentions"] = 0
            
    else:
        prediction["survey_mentions"] = None
    
    return prediction

In [72]:
result = get_wifi_prediction(
    location="LIBRARY",
    time="9:00 PM",
    crowd=3
)

print(result)

{'location': 'LIBRARY', 'time': '9:00 PM', 'hour': 21, 'session': 'NIGHT', 'crowd': 3, 'historical_signal_dbm': -63.5, 'prediction': 1, 'poor_connectivity_probability': np.float64(0.8247653813775464), 'risk': 'HIGH', 'median_download_mbps': np.float64(0.34), 'ping_failure_rate': np.float64(100.0), 'median_crowd': np.float64(3.0), 'survey_category': 'LIBRARY', 'survey_mentions': 7}


In [75]:
def generate_wifi_response(result):

    if "error" in result:
        return result["error"]

    probability = result["poor_connectivity_probability"] * 100

    location = result["location"]
    time = result["time"]
    crowd = result["crowd"]
    risk = result["risk"]

    signal = result["historical_signal_dbm"]
    download = result["median_download_mbps"]
    ping_failure = result["ping_failure_rate"]

    survey_mentions = result.get("survey_mentions")
    survey_category = result.get("survey_category")

    # -----------------------------
    # Main prediction
    # -----------------------------

    response = f"""
 Wi-Fi Connectivity Assessment

Location: {location}
Time: {time}
Expected crowd: {crowd}

Connectivity Risk: {risk}
Estimated probability of poor connectivity: {probability:.1f}%

Network evidence:
• Median signal strength: {signal:.1f} dBm
• Median download speed: {download:.2f} Mbps
• Ping failure rate: {ping_failure:.1f}%
"""

    # -----------------------------
    # Survey evidence
    # -----------------------------

    if (
        survey_mentions is not None
        and pd.notna(survey_mentions)
        and survey_mentions > 0
    ):
        response += f"""
Student survey evidence:
• {survey_mentions} survey responses were associated with the "{survey_category}" category.
"""

    else:
        response += """
Student survey evidence:
• No directly mapped survey category is currently available for this location.
"""

    # -----------------------------
    # Interpretation
    # -----------------------------

    if risk == "HIGH":

        response += """
Interpretation:
The selected location and crowd conditions correspond to a high predicted
risk of poor connectivity based on the network measurements collected.
"""

    elif risk == "MEDIUM":

        response += """
Interpretation:
The selected conditions indicate a moderate predicted connectivity risk.
"""

    else:

        response += """
Interpretation:
The selected conditions indicate a relatively lower predicted connectivity risk.
"""

    return response

In [76]:
result = get_wifi_prediction(
    location="LIBRARY",
    time="9:00 PM",
    crowd=3
)

print(generate_wifi_response(result))


 Wi-Fi Connectivity Assessment

Location: LIBRARY
Time: 9:00 PM
Expected crowd: 3

Connectivity Risk: HIGH
Estimated probability of poor connectivity: 82.5%

Network evidence:
• Median signal strength: -63.5 dBm
• Median download speed: 0.34 Mbps
• Ping failure rate: 100.0%

Student survey evidence:
• 7 survey responses were associated with the "LIBRARY" category.

Interpretation:
The selected location and crowd conditions correspond to a high predicted
risk of poor connectivity based on the network measurements collected.

